# MVP con IA en 2 Semanas

Desde el system prompt hasta el primer usuario de pago.

In [ ]:
import anthropic
import json
from pathlib import Path
from datetime import datetime

client = anthropic.Anthropic()
print("Taller de MVP listo")

## Semana 1, Día 1: Diseñar y testear el system prompt

In [ ]:
# Define aquí el system prompt de tu producto
SYSTEM_V1 = """Eres un asistente especializado en análisis de contratos para PYMEs españolas.
Tu función es identificar cláusulas problemáticas, explicarlas en lenguaje claro
y sugerir alternativas más favorables para el cliente.

REGLAS:
- Explica cada cláusula problemática en una frase
- Indica el riesgo: BAJO / MEDIO / ALTO
- Sugiere siempre una alternativa concreta
- Si no hay problemas, dilo explícitamente
- Nunca des asesoramiento jurídico vinculante"""


def evaluar_prompt(system: str, casos: list[dict]) -> dict:
    """Evalúa un system prompt con casos de prueba y devuelve métricas."""
    resultados = []
    tokens_totales = 0

    for caso in casos:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            system=system,
            messages=[{"role": "user", "content": caso["input"]}]
        )
        output = resp.content[0].text
        tokens_totales += resp.usage.input_tokens + resp.usage.output_tokens

        # Autoevaluar con Claude si el output cumple el criterio esperado
        eval_resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=50,
            messages=[{
                "role": "user",
                "content": f"""¿Este output cumple el criterio? Responde solo SÍ o NO.
Criterio: {caso.get('esperado', 'respuesta útil y concisa')}
Output: {output[:300]}"""
            }]
        )
        cumple = "sí" in eval_resp.content[0].text.lower()
        resultados.append({"input": caso["input"][:60], "output": output[:150], "cumple": cumple})

    tasa_exito = sum(1 for r in resultados if r["cumple"]) / len(resultados)
    coste_total = tokens_totales / 1_000_000 * 1.20

    return {
        "tasa_exito_pct": round(tasa_exito * 100, 1),
        "tokens_totales": tokens_totales,
        "coste_total_usd": round(coste_total, 4),
        "coste_por_caso_usd": round(coste_total / len(casos), 5),
        "casos": resultados
    }


CASOS_TEST = [
    {"input": "Cláusula: El proveedor no asume responsabilidad por ningún daño directo o indirecto.",
     "esperado": "identifica riesgo ALTO y sugiere alternativa"},
    {"input": "Cláusula: El precio se revisará anualmente según el IPC del INE.",
     "esperado": "identifica como cláusula razonable, riesgo BAJO"},
    {"input": "Cláusula: Cualquier disputa se resolverá en tribunales de Delaware.",
     "esperado": "identifica problema de jurisdicción extranjera"},
    {"input": "Cláusula: El contrato se renovará automáticamente cada año sin notificación previa.",
     "esperado": "identifica riesgo de renovación automática sin aviso"},
]

print("Evaluando system prompt v1.0...")
resultado_v1 = evaluar_prompt(SYSTEM_V1, CASOS_TEST)

print(f"\nTasa de éxito: {resultado_v1['tasa_exito_pct']}%")
print(f"Coste por caso: ${resultado_v1['coste_por_caso_usd']:.5f}")
print(f"\nDetalle por caso:")
for r in resultado_v1["casos"]:
    icono = "✅" if r["cumple"] else "❌"
    print(f"  {icono} {r['input']}")

## Gestión de versiones del prompt

In [ ]:
class GestorDePrompts:
    def __init__(self, directorio: str = "prompts_mvp"):
        self.dir = Path(directorio)
        self.dir.mkdir(exist_ok=True)

    def guardar(self, nombre: str, contenido: str, version: str, notas: str = ""):
        meta = {"nombre": nombre, "version": version,
                "fecha": datetime.now().isoformat(), "notas": notas}
        (self.dir / f"{nombre}_v{version}.txt").write_text(contenido, encoding="utf-8")
        (self.dir / f"{nombre}_v{version}.json").write_text(
            json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print(f"💾 Guardado: {nombre} v{version}")

    def cargar(self, nombre: str, version: str) -> str:
        ruta = self.dir / f"{nombre}_v{version}.txt"
        return ruta.read_text(encoding="utf-8") if ruta.exists() else ""

    def listar(self, nombre: str) -> list:
        return [
            json.loads(p.read_text())
            for p in sorted(self.dir.glob(f"{nombre}_v*.json"))
        ]


gestor = GestorDePrompts()
gestor.guardar("analizador_contratos", SYSTEM_V1, "1.0", "Versión inicial")

# Versión mejorada basada en el análisis de fallos
SYSTEM_V2 = SYSTEM_V1 + """

CASOS ESPECIALES:
- Si el contrato menciona renovación automática sin notificación, marca como ALTO
- Si la jurisdicción es extranjera para una empresa española, marca como ALTO
- Para contratos de trabajo, indica que aplica el ET español"""

gestor.guardar("analizador_contratos", SYSTEM_V2, "1.1", "Añadidos casos especiales de jurisdicción y renovación")

print("\nVersiones guardadas:")
for v in gestor.listar("analizador_contratos"):
    print(f"  v{v['version']} ({v['fecha'][:10]}): {v['notas']}")

## Comparar versiones del prompt

In [ ]:
print("Comparando v1.0 vs v1.1...")
resultado_v2 = evaluar_prompt(SYSTEM_V2, CASOS_TEST)

print(f"\n{'Métrica':<30} {'v1.0':>10} {'v1.1':>10} {'Mejora':>10}")
print("-" * 60)
print(f"{'Tasa de éxito':<30} {resultado_v1['tasa_exito_pct']:>9}% {resultado_v2['tasa_exito_pct']:>9}% {resultado_v2['tasa_exito_pct']-resultado_v1['tasa_exito_pct']:>+9.1f}%")
print(f"{'Coste/caso (USD)':<30} ${resultado_v1['coste_por_caso_usd']:>8.5f} ${resultado_v2['coste_por_caso_usd']:>8.5f}")

print("\n¿Qué versión usar?")
if resultado_v2['tasa_exito_pct'] > resultado_v1['tasa_exito_pct']:
    print("✅ v1.1 — mejora la calidad")
else:
    print("⚠️  v1.0 — sin mejora clara, revisar los casos de prueba")

## Pipeline completo: del texto al análisis

In [ ]:
CONTRATO_DEMO = """
CONTRATO DE SERVICIOS DE DESARROLLO WEB

CLÁUSULA 1 - OBJETO: El proveedor desarrollará la plataforma descrita en el Anexo A.
CLÁUSULA 2 - PRECIO: 12.000€ pagaderos en 3 tramos según hitos.
CLÁUSULA 3 - PROPIEDAD: Todo el código desarrollado será propiedad del Proveedor.
  El Cliente recibe licencia de uso no exclusiva e intransferible.
CLÁUSULA 4 - RESPONSABILIDAD: La responsabilidad máxima del Proveedor no excederá
  el importe de un mes de servicio (400€).
CLÁUSULA 5 - RENOVACIÓN: El contrato se renovará automáticamente cada año
  salvo aviso con 7 días de antelación.
CLÁUSULA 6 - JURISDICCIÓN: Cualquier disputa se resolverá en los tribunales
  de la ciudad de Dublin, Irlanda.
"""


def pipeline_analisis(texto: str) -> dict:
    """Pipeline completo: extraer cláusulas → analizar → resumen ejecutivo."""

    # Paso 1: Extraer cláusulas
    r_extraccion = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": f"""Extrae las cláusulas de este contrato.
JSON: {{"clausulas": [{{"num": 1, "titulo": "...", "texto": "..."}}]}}

{texto}"""
        }]
    )
    txt = r_extraccion.content[0].text
    if "```" in txt:
        txt = txt.split("```")[1].lstrip("json\n")
    try:
        clausulas = json.loads(txt).get("clausulas", [])
    except json.JSONDecodeError:
        clausulas = []

    # Paso 2: Analizar cada cláusula
    analisis = []
    for c in clausulas:
        r_analisis = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            system=SYSTEM_V2,
            messages=[{"role": "user", "content": f"{c.get('titulo', '')}: {c.get('texto', '')[:400]}"}]
        )
        analisis.append({"clausula": c.get("titulo", f"Cláusula {c.get('num', '?')}"),
                          "analisis": r_analisis.content[0].text})

    # Paso 3: Resumen ejecutivo
    texto_analisis = "\n".join(f"- {a['clausula']}: {a['analisis'][:80]}" for a in analisis)
    r_resumen = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""Genera un resumen ejecutivo en 3 puntos para el empresario:
{texto_analisis}
Formato: 1. [problema más urgente] 2. [segundo riesgo] 3. [recomendación de acción]"""
        }]
    )

    return {"clausulas_analizadas": len(clausulas),
            "detalle": analisis,
            "resumen_ejecutivo": r_resumen.content[0].text}


print("Analizando contrato demo...")
resultado = pipeline_analisis(CONTRATO_DEMO)

print(f"\nCláusulas analizadas: {resultado['clausulas_analizadas']}")
print("\nDetalle:")
for d in resultado["detalle"]:
    print(f"  • {d['clausula']}: {d['analisis'][:100]}...")

print("\n=== RESUMEN EJECUTIVO ===")
print(resultado["resumen_ejecutivo"])

## Calcular precio del producto

In [ ]:
def calcular_precio(coste_tokens_mes_usd: float, margen: float = 0.70, overhead: float = 1.30) -> dict:
    coste_total = coste_tokens_mes_usd * overhead
    precio_min = coste_total / (1 - margen)
    tiers = [9, 19, 29, 49, 79, 99, 149, 199]
    precio_recomendado = next((t for t in tiers if t >= precio_min), tiers[-1])
    return {
        "coste_tokens_usd": round(coste_tokens_mes_usd, 3),
        "coste_con_overhead_usd": round(coste_total, 3),
        "precio_minimo_usd": round(precio_min, 2),
        "precio_recomendado_usd": precio_recomendado,
        "margen_real_pct": round((1 - coste_total / precio_recomendado) * 100, 1)
    }


print("ESTRUCTURA DE PRECIOS")
print("=" * 55)
escenarios = [
    ("Plan Starter (20 análisis/mes)", 20 * 1200 / 1_000_000 * 1.20),
    ("Plan Pro (100 análisis/mes)", 100 * 1200 / 1_000_000 * 1.20),
    ("Plan Business (500 análisis/mes)", 500 * 1200 / 1_000_000 * 1.20),
]

for nombre, coste in escenarios:
    p = calcular_precio(coste)
    print(f"\n{nombre}")
    print(f"  Coste API: ${p['coste_tokens_usd']:.3f} | Precio recomendado: ${p['precio_recomendado_usd']}/mes")
    print(f"  Margen bruto: {p['margen_real_pct']}%")